cells update:
1. modified
2. unchanged
3. unchanged
4. unchanged
5. added new ver
6. added new ver
7. added new ver
8. unchanged
9. added new ver


details for each cell changes is in these markdown cells

cell 1: added lines for constant value columns + ensuring only the 15 sensors are picked for the model

In [1]:
# ── CELL 1: Imports & Configuration ───────────────────────────────────────
# Sets all paths, constants, and feature/drop column lists.
# MAX_LEN=1028 is the verified minimum timestep count across all normal samples.
# MANUAL_DROP removes meta/label columns — only raw sensor signals are used.
import pandas as pd
import numpy as np
import pickle
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from pathlib import Path

df = pd.read_parquet("voraus-ad-dataset-100hz.parquet")

# ── Paths ──────────────────────────────────────────────────────────────────
RAW_PATH    = Path("voraus-ad-dataset-100hz.parquet")
OUTPUT_DIR  = Path("data/processed")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Reproducibility ────────────────────────────────────────────────────────
RANDOM_STATE = 42

# ── Sequence config ────────────────────────────────────────────────────────
MAX_LEN     = 1028   # minimum timesteps across all normal samples (verified)

# ── Features to DROP ───────────────────────────────────────────────────────
MANUAL_DROP = ['time', 'sample', 'anomaly', 'category', 'setting', 'action', 'active']

# Potential features with constant columns
temp_features = [c for c in df.columns if c not in MANUAL_DROP]

# find the constant columns where sensors never change
constant_cols = [col for col in df[temp_features].columns if df[col].nunique() <= 1]

# combining all columns to be dropped
DROP_COLS = MANUAL_DROP + constant_cols

# Finalised sensors for the model (to ensure the vector has 300 features: 20 rows *  15 sensors)
SENSORS = [
    'motor_torque_1', 'motor_torque_2', 'motor_torque_3', 
    'motor_torque_4', 'motor_torque_5', 'motor_torque_6',
    'joint_velocity_1', 'joint_velocity_2', 'joint_velocity_3', 
    'joint_velocity_4', 'joint_velocity_5', 'joint_velocity_6',
    'robot_current', 'robot_voltage', 'system_current'
]

# Final list of features that the model will see
FEATURE_COLS = [c for c in SENSORS if c not in DROP_COLS]
print(f"Number of features selected for the vector: {len(FEATURE_COLS)}")

# Remaining 130 columns are sensor features fed to the LSTM

Number of features selected for the vector: 15


In [2]:
# ── CELL 2: Load & Verify Raw Data ────────────────────────────────────────
# Loads the parquet file and asserts expected shape and sample count.
# Fails fast here if the raw file is missing or malformed.df = pd.read_parquet(RAW_PATH)

assert df.shape[1] == 137, "Column count mismatch — check raw file"
assert df['sample'].nunique() == 2122, "Sample count mismatch"
assert df['anomaly'].dtype == bool

print(f"Loaded: {df.shape}")
print(f"Samples: {df['sample'].nunique()}")
print(f"Features after dropping columns: {df.shape[1] - len(DROP_COLS)}")

Loaded: (2321690, 137)
Samples: 2122
Features after dropping columns: 129


In [3]:
# ── CELL 3: Build Sample Metadata ─────────────────────────────────────────
# Aggregates per-timestep data into one row per sample.
# Separates normal (1,367) and anomalous (755) samples.
# normal_meta is shuffled here — this is the only source of randomness.
meta = (
    df.groupby('sample')
    .agg(anomaly=('anomaly', 'first'),
         category=('category', 'first'),
         setting=('setting', 'first'),
         n_timesteps=('time', 'count'))
    .reset_index()
)

normal_meta  = meta[meta['anomaly'] == False].sample(frac=1, random_state=RANDOM_STATE)
anomaly_meta = meta[meta['anomaly'] == True]

print(f"Normal: {len(normal_meta)} | Anomaly: {len(anomaly_meta)}")

Normal: 1367 | Anomaly: 755


In [4]:
# ── CELL 4: Train / Val / Test Split ──────────────────────────────────────
# Splits normal samples 70/15/15 using RANDOM_STATE=42 for reproducibility.
# Anomalies are held out entirely and only added to the test set.
# This mirrors real deployment: the model never sees anomalies during training.
train_meta, temp_meta = train_test_split(normal_meta, test_size=0.30, random_state=RANDOM_STATE)
val_meta, test_normal_meta = train_test_split(temp_meta, test_size=0.50, random_state=RANDOM_STATE)

# Test set = normal holdout + ALL anomalies
test_meta = pd.concat([test_normal_meta, anomaly_meta]).sample(frac=1, random_state=RANDOM_STATE)

print(f"Train : {len(train_meta)} normal")
print(f"Val   : {len(val_meta)} normal")
print(f"Test  : {len(test_meta)} ({len(test_normal_meta)} normal + {len(anomaly_meta)} anomaly)")

Train : 956 normal
Val   : 205 normal
Test  : 961 (206 normal + 755 anomaly)


original cell #5 was not modified and used

see below for new cell 5 reason

In [14]:
# ── CELL 5: Sequence Loader ────────────────────────────────────────────────
# load_sequence() tail-truncates each sample to MAX_LEN timesteps.
# Zero-padding is only triggered for 8 anomalous samples (length < 1028)
# which never appear in train or val — so training data is never padded.
FEATURE_COLS = [c for c in df.columns if c not in DROP_COLS]

def load_sequence(sample_id: int, max_len: int = MAX_LEN) -> np.ndarray:
    """
    Load one sample's sensor data, tail-truncate (or zero-pad if shorter).
    Returns shape: (max_len, n_features)
    """
    seq = df[df['sample'] == sample_id][FEATURE_COLS].values  # (T, 130)

    if len(seq) >= max_len:
        return seq[-max_len:]          # keep most recent timesteps
    else:
        # Only affects 8 anomalous samples — never seen during training
        pad = np.zeros((max_len - len(seq), seq.shape[1]))
        return np.vstack([pad, seq])   # pre-pad to preserve recency


def build_array(sample_ids: pd.Series) -> np.ndarray:
    """Stack sequences into (N, MAX_LEN, n_features) array."""
    return np.stack([load_sequence(sid) for sid in sample_ids], axis=0)

new cell 5: sliding window loader method
- breaks long cycles into many small snapshots of timesteps (20) unlike above cell 5

why i think this is better

current cell 5 loads the entire 1028 timesteps so the input shape is (1028, 130) which is over 100k numbers per sample
- too big for MLP autoencoder that was proposed in the project proposal

In [5]:
# ── CELL 5: Sliding Window Loader ──────────────────────────────────────────
# Breaks long cycles into many small snapshots.
# This uses a window of 20 timesteps across 15 sensors = 300 features. (20 * 15 = 300)

def build_windows(sample_ids, dataframe, sensors, window_size=20, step=5):
    """
    Chunks cycles into overlapping windows.
    Returns: X, y 
    where X = (matrix of 300-length vectors)
          y = (anomaly labels)
    """

    X_windows = []
    y_labels = [] 
    
    for sid in sample_ids:
        # Get all sensor data for this specific cycle
        cycle_data = dataframe[dataframe['sample'] == sid][sensors].values
        # Check if this cycle is marked as anomaly in the metadata
        is_anomaly = dataframe[dataframe['sample'] == sid]['anomaly'].iloc[0]
        
        # Slide the window across the time series
        # range(start, stop, step) -> when its step=5, it reduces overlap to save memory/time
        for i in range(0, len(cycle_data) - window_size, step): 
            window = cycle_data[i : i + window_size]
            
            # Flatten 2D into 1D -> (20, 15) becomes (300,)
            X_windows.append(window.flatten()) 
            y_labels.append(1 if is_anomaly else 0)
            
    return np.array(X_windows), np.array(y_labels)

print("Sliding window function defined.")

Sliding window function defined.


new cell 6 - connected to cell 5

In [6]:
# ── CELL 6: Build Raw Arrays ───────────────────────────────────────────────
print("Building 300-feature windows (this may take a minute)...")

X_train_raw, _ = build_windows(train_meta['sample'], df, FEATURE_COLS)
X_val_raw, _   = build_windows(val_meta['sample'], df, FEATURE_COLS)
X_test_raw, y_test = build_windows(test_meta['sample'], df, FEATURE_COLS)

print(f"X_train Shape: {X_train_raw.shape}") # Should show (N, 300)

Building 300-feature windows (this may take a minute)...
X_train Shape: (204854, 300)


prev cell 6 (not modified and not used)

In [17]:
# ── CELL 6: Build Raw Arrays ───────────────────────────────────────────────
# Stacks all sequences into 3D numpy arrays (N, 1028, 130).
# y_test contains binary labels (0=normal, 1=anomaly) for evaluation only.
# Expected output: X_train (956,), X_val (205,), X_test (961,) samples.
print("Building arrays (this may take a moment)...")

X_train_raw = build_array(train_meta['sample'])
X_val_raw   = build_array(val_meta['sample'])
X_test_raw  = build_array(test_meta['sample'])

# Labels for test evaluation only
y_test = test_meta['anomaly'].astype(int).values  # 0=normal, 1=anomaly

print(f"X_train : {X_train_raw.shape}")   # (956, 1028, 130)
print(f"X_val   : {X_val_raw.shape}")     # (205, 1028, 130)
print(f"X_test  : {X_test_raw.shape}")    # (961, 1028, 130)
print(f"y_test  : {y_test.shape} | anomaly rate: {y_test.mean():.1%}")

Building arrays (this may take a moment)...
X_train : (956, 1028, 129)
X_val   : (205, 1028, 129)
X_test  : (961, 1028, 129)
y_test  : (961,) | anomaly rate: 78.6%


prev cell 7 (not modified and not used)

In [18]:
# ── CELL 7: Fit & Apply Scaler ─────────────────────────────────────────────
# StandardScaler is fit ONLY on X_train_raw to prevent leakage.
# Val and test sets are transformed using train statistics.
# Reshape to (N*T, F) for fitting, then back to (N, T, F) after.

# Flatten to (N*T, F) to fit, then reshape back
n_train, T, F = X_train_raw.shape

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_raw.reshape(-1, F)).reshape(n_train, T, F)
X_val   = scaler.transform(X_val_raw.reshape(-1, F)).reshape(X_val_raw.shape)
X_test  = scaler.transform(X_test_raw.reshape(-1, F)).reshape(X_test_raw.shape)

print("Scaler fit on train only — val/test transformed, not fit")

Scaler fit on train only — val/test transformed, not fit


new cell 7: float32 conversion for pytorch compatibility

In [7]:
# ── CELL 7: Fit & Apply Scaler ─────────────────────────────────────────────
# StandardScaler is fit ONLY on X_train_raw to prevent leakage.
# Val and test sets are transformed using train statistics.=

scaler = StandardScaler()

# Transform all sets and force them to float32 for PyTorch compatibility
X_train = scaler.fit_transform(X_train_raw).astype('float32')
X_val   = scaler.transform(X_val_raw).astype('float32')
X_test  = scaler.transform(X_test_raw).astype('float32')

print(f"Data scaled and converted to {X_train.dtype}")
print(f"Final train Shape: {X_train.shape}")

Data scaled and converted to float32
Final train Shape: (204854, 300)


In [8]:
# ── CELL 8: Save Processed Data ───────────────────────────────────────────
# Saves scaled arrays to splits.npz and scaler to scaler.pkl.
# Meta parquets saved for traceability (which sample IDs are in each split).
# All files written to data/processed/ which is NOT tracked by git.

# Processed arrays
np.savez_compressed(
    OUTPUT_DIR / "splits.npz",
    X_train=X_train,
    X_val=X_val,
    X_test=X_test,
    y_test=y_test
)

# Scaler (needed for inverse_transform at inference)
with open(OUTPUT_DIR / "scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)

# Split metadata (for traceability — which sample IDs are in each split)
train_meta.to_parquet(OUTPUT_DIR / "train_meta.parquet", index=False)
val_meta.to_parquet(OUTPUT_DIR / "val_meta.parquet", index=False)
test_meta.to_parquet(OUTPUT_DIR / "test_meta.parquet", index=False)

print("Saved:")
print(f"  {OUTPUT_DIR}/splits.npz")
print(f"  {OUTPUT_DIR}/scaler.pkl")
print(f"  {OUTPUT_DIR}/train_meta.parquet")
print(f"  {OUTPUT_DIR}/val_meta.parquet")
print(f"  {OUTPUT_DIR}/test_meta.parquet")

Saved:
  data\processed/splits.npz
  data\processed/scaler.pkl
  data\processed/train_meta.parquet
  data\processed/val_meta.parquet
  data\processed/test_meta.parquet


new cell 9 to verify integrity

In [9]:
# ── CELL 9: Reload & Verify ───────────────────────────────────────────────
# This verifies the sliding window data is correct and ready for the MLP.

# Reload the compressed arrays
data   = np.load(OUTPUT_DIR / "splits.npz")
X_tr   = data['X_train']
X_v    = data['X_val']
X_te   = data['X_test']
y_te   = data['y_test']

# Reload the scaler to ensure its readable
with open(OUTPUT_DIR / "scaler.pkl", "rb") as f:
    scaler_loaded = pickle.load(f)

# Check that the input dimension is exactly 300
assert X_tr.shape[1] == 300, f"Expected 300 features, got {X_tr.shape[1]}"

# Check that X_train and X_val have no anomalies (y should be 0 for all)
# Since X_train/X_val only came from normal_meta, we just check shape logic
assert X_tr.shape[0] > 100000, "X_train seems too small, check window step size"

# Check for NaNs
assert not np.isnan(X_tr).any(), "Found NaNs in X_train"

# Final Output Report
print("✓ All checks passed")
print(f"  Training Windows   : {X_tr.shape[0]} (All Normal)")
print(f"  Validation Windows : {X_v.shape[0]}")
print(f"  Test Windows       : {X_te.shape[0]}")
print(f"  Anomaly Rate in Test: {y_te.mean():.1%}")
print(f"  Data Type          : {X_tr.dtype}")

✓ All checks passed
  Training Windows   : 204854 (All Normal)
  Validation Windows : 43865
  Test Windows       : 208075
  Anomaly Rate in Test: 78.8%
  Data Type          : float32


In [23]:
# ── CELL 9: Reload & Verify ───────────────────────────────────────────────
# Reloads all saved files from disk and checks shapes and label counts.
# You should see "✓ All checks passed" before proceeding.
# This cell can be run independently to verify data integrity at any time.

# Run this cell to verify integrity
data   = np.load(OUTPUT_DIR / "splits.npz")
X_tr   = data['X_train']
X_v    = data['X_val']
X_te   = data['X_test']
y_te   = data['y_test']

with open(OUTPUT_DIR / "scaler.pkl", "rb") as f:
    scaler_loaded = pickle.load(f)

assert X_tr.shape  == (len(train_meta), MAX_LEN, len(FEATURE_COLS))
assert X_te.shape[0] == len(test_meta)
assert y_te.sum()  == 755   # all anomalies accounted for

print("✓ All checks passed")
print(f"  X_train : {X_tr.shape}")
print(f"  X_val   : {X_v.shape}")
print(f"  X_test  : {X_te.shape} | anomaly rate: {y_te.mean():.1%}")

✓ All checks passed
  X_train : (956, 1028, 130)
  X_val   : (205, 1028, 130)
  X_test  : (961, 1028, 130) | anomaly rate: 78.6%
